# Tutorial 4: RAG Q&A Generation

This notebook shows how to generate grounded question-answer pairs from a source text chunk using `RAGQAGenerator`.

These Q&A pairs are useful for evaluating retrieval-augmented generation (RAG) systems later. Each sample contains a question and answer that should be answerable from the provided chunk only.

By the end of this tutorial you will know how to:
1. Generate RAG Q&A samples from a text chunk
2. Inspect the returned DataFrame and metadata
3. Run the generator with a mock provider or OpenAI
4. Save Q&A samples for future evaluation

## Installation

Install the library with the OpenAI provider if you want to call a live model:

In [ ]:
# Uncomment to install from PyPI
# !pip install synthetictext[openai]

## Source Chunk

RAG Q&A generation starts with a chunk of text. The generator should only create questions whose answers are present in this chunk.

In [ ]:
chunk = """
Synthetictext is a Python library for generating synthetic text data with LLMs.
It supports classification data generation through strategies such as direct generation,
paraphrasing, contrastive pairs, backtranslation, and pivot translation.

The RAG Q&A generator creates question-answer pairs from a provided source chunk.
Each answer should be grounded in the chunk rather than outside knowledge. The output
is returned as a pandas DataFrame with question, answer, source, and metadata columns.
""".strip()

print(chunk)

## Run Without an API Key

For tutorials, tests, and examples, you can pass any object that implements the `BaseLLMProvider.generate(...)` interface. This mock provider returns deterministic JSON so the notebook can run without a live API call.

In [ ]:
from synthetictext import BaseLLMProvider, RAGQAGenerator


class MockQALLMProvider(BaseLLMProvider):
    def generate(
        self,
        prompt: str,
        *,
        model=None,
        temperature=0.9,
        max_tokens=250,
        system_prompt=None,
        response_format=None,
    ) -> str:
        # RAGQAGenerator asks OpenAI-compatible providers for JSON mode via response_format.
        assert response_format == {"type": "json_object"}
        return '''{
            "qa_pairs": [
                {
                    "question": "What type of library is synthetictext?",
                    "answer": "Synthetictext is a Python library for generating synthetic text data with LLMs."
                },
                {
                    "question": "What does the RAG Q&A generator create?",
                    "answer": "It creates question-answer pairs from a provided source chunk."
                },
                {
                    "question": "What format does the RAG Q&A generator return?",
                    "answer": "It returns a pandas DataFrame with question, answer, source, and metadata columns."
                }
            ]
        }'''


In [ ]:
qa_generator = RAGQAGenerator(llm_provider=MockQALLMProvider())
qa_df = qa_generator.generate(chunk=chunk, num_samples=3, language="English")

qa_df

## Inspect Metadata

The source chunk is not duplicated as a top-level column. Instead, each row carries metadata with the generation language, a stable chunk hash, and a short preview. This makes it easy to trace samples back to the chunk without bloating the main table.

In [ ]:
qa_df.loc[0, "metadata"]

## Use OpenAI

For real generation, use the OpenAI provider shorthand. `RAGQAGenerator` requests OpenAI JSON mode so the model is constrained to return a JSON object.

In [ ]:
import os

# Uncomment after setting OPENAI_API_KEY in your environment.
# assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this cell"
# openai_qa = RAGQAGenerator(llm_provider="openai", llm_model="gpt-4o-mini")
# openai_df = openai_qa.generate(chunk=chunk, num_samples=5, language="English")
# openai_df

## Save for Evaluation

The returned DataFrame can be saved directly to CSV and used later when building a RAG evaluation harness.

In [ ]:
output_path = "rag_qa_samples.csv"
qa_df.to_csv(output_path, index=False)
print(f"Saved {len(qa_df)} samples to {output_path}")

## CLI Equivalent

You can also generate Q&A pairs from a text file using the CLI:

```bash
synthetictext rag-qa --input chunk.txt --num-samples 5 --language English --output qa.csv
```